In [1]:
pip install yfinance pandas numpy matplotlib mplfinance plotly

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 30.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 51.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 19.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 42.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 25.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 11.5 MB/s  0:00:00 eta 0:00:01
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15635 sha256=807ac0efb86cbc0b8333d71783ec41e8dfe7ef62df57e4a9f836d2898f314cf2
  Stored in directory: /Users/lol/Library/Caches/pip/wheels/7e/62/f9/20d7dbb144b6f563edab8e3a7fda71d976870cd41972035cdd
Successfully built multitask

In [6]:
import plotly.io as pio
pio.renderers.default = "notebook"

import numpy as np
import pandas as pd
import yfinance as yf
import mplfinance as mpf
import matplotlib.pyplot as plt

from datetime import datetime
from pathlib import Path
import webbrowser


def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """
    RSI 계산 (Wilder 방식에 가까운 EWM 사용)
    """
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def add_indicators(
    df: pd.DataFrame,
    bb_period: int = 20,
    bb_std: float = 2.0,
    rsi_period: int = 14,
    squeeze_lookback: int = 120,
    squeeze_percentile: float = 0.2,
) -> pd.DataFrame:
    """
    볼린저 밴드, RSI, 스퀴즈, 전략 신호 계산
    """
    df = df.copy()

    # Bollinger Bands
    df["MB"] = df["Close"].rolling(bb_period).mean()
    df["STD"] = df["Close"].rolling(bb_period).std(ddof=0)
    df["UB"] = df["MB"] + bb_std * df["STD"]
    df["LB"] = df["MB"] - bb_std * df["STD"]

    # Bollinger Band Width
    df["BBW"] = (df["UB"] - df["LB"]) / df["MB"] * 100

    # RSI
    df["RSI"] = compute_rsi(df["Close"], period=rsi_period)

    # Squeeze threshold
    df["BBW_Q"] = (
        df["BBW"]
        .rolling(squeeze_lookback)
        .apply(lambda x: np.nanpercentile(x, squeeze_percentile * 100), raw=True)
    )
    df["SQUEEZE"] = df["BBW"] <= df["BBW_Q"]

    # Previous values
    df["PrevClose"] = df["Close"].shift(1)
    df["PrevLow"] = df["Low"].shift(1)
    df["PrevHigh"] = df["High"].shift(1)

    # Strategy signals
    lower_touch = (df["Low"] <= df["LB"]) | (df["PrevLow"] <= df["LB"].shift(1))
    bullish_confirm = df["Close"] > df["PrevClose"]
    df["BUY_REBOUND"] = lower_touch & (df["RSI"] <= 30) & bullish_confirm

    upper_touch = (df["High"] >= df["UB"]) | (df["PrevHigh"] >= df["UB"].shift(1))
    bearish_confirm = df["Close"] < df["PrevClose"]
    df["SELL_REVERSION"] = upper_touch & (df["RSI"] >= 70) & bearish_confirm

    prev_squeeze = df["SQUEEZE"].shift(1).fillna(False)
    df["BREAKOUT_UP"] = prev_squeeze & (df["Close"] > df["UB"]) & (df["RSI"] > 50)
    df["BREAKOUT_DOWN"] = prev_squeeze & (df["Close"] < df["LB"]) & (df["RSI"] < 50)

    return df


def download_data(
    ticker: str,
    period: str = "3y",
    interval: str = "1d",
) -> pd.DataFrame:
    """
    Yahoo Finance 데이터 다운로드
    """
    df = yf.download(
        ticker,
        period=period,
        interval=interval,
        auto_adjust=False,
        progress=False,
        group_by="column",
        multi_level_index=False,
    )

    if df.empty:
        raise ValueError(f"데이터를 가져오지 못했습니다: {ticker}")

    expected_cols = ["Open", "High", "Low", "Close", "Volume"]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    df = df[expected_cols].dropna().copy()
    return df


def save_csv_with_today(df: pd.DataFrame, ticker: str, output_dir: Path) -> Path:
    """
    YYYYMMDD_ticker.csv 형태로 저장
    """
    today_str = datetime.now().strftime("%Y%m%d")
    filename = f"{today_str}_{ticker.lower()}.csv"
    filepath = output_dir / filename

    df_to_save = df.copy().reset_index()

    # 인덱스 날짜 컬럼 이름 정리
    first_col = df_to_save.columns[0]
    if first_col != "Date":
        df_to_save.rename(columns={first_col: "Date"}, inplace=True)

    df_to_save.to_csv(filepath, index=False, encoding="utf-8-sig")
    print(f"CSV 저장 완료: {filepath}")
    return filepath


def make_marker_series(df: pd.DataFrame) -> dict:
    buy_marker = pd.Series(np.nan, index=df.index)
    sell_marker = pd.Series(np.nan, index=df.index)
    breakout_up_marker = pd.Series(np.nan, index=df.index)
    breakout_down_marker = pd.Series(np.nan, index=df.index)

    buy_marker[df["BUY_REBOUND"]] = df.loc[df["BUY_REBOUND"], "Low"] * 0.985
    sell_marker[df["SELL_REVERSION"]] = df.loc[df["SELL_REVERSION"], "High"] * 1.015
    breakout_up_marker[df["BREAKOUT_UP"]] = df.loc[df["BREAKOUT_UP"], "Low"] * 0.975
    breakout_down_marker[df["BREAKOUT_DOWN"]] = df.loc[df["BREAKOUT_DOWN"], "High"] * 1.025

    return {
        "buy_marker": buy_marker,
        "sell_marker": sell_marker,
        "breakout_up_marker": breakout_up_marker,
        "breakout_down_marker": breakout_down_marker,
    }


def save_static_chart(
    df: pd.DataFrame,
    ticker: str,
    output_dir: Path,
    bb_period: int,
    bb_std: float,
    rsi_period: int,
    volume: bool = True,
) -> Path:
    """
    mplfinance 정적 PNG 차트 저장
    """
    markers = make_marker_series(df)

    rsi_70 = pd.Series(70, index=df.index)
    rsi_50 = pd.Series(50, index=df.index)
    rsi_30 = pd.Series(30, index=df.index)

    addplots = [
        mpf.make_addplot(df["UB"], panel=0, width=1),
        mpf.make_addplot(df["MB"], panel=0, width=1),
        mpf.make_addplot(df["LB"], panel=0, width=1),

        mpf.make_addplot(
            markers["buy_marker"],
            panel=0,
            type="scatter",
            marker="^",
            markersize=100,
        ),
        mpf.make_addplot(
            markers["sell_marker"],
            panel=0,
            type="scatter",
            marker="v",
            markersize=100,
        ),
        mpf.make_addplot(
            markers["breakout_up_marker"],
            panel=0,
            type="scatter",
            marker="*",
            markersize=160,
        ),
        mpf.make_addplot(
            markers["breakout_down_marker"],
            panel=0,
            type="scatter",
            marker="X",
            markersize=120,
        ),

        mpf.make_addplot(df["RSI"], panel=2, ylabel="RSI", width=1),
        mpf.make_addplot(rsi_70, panel=2, width=0.8, linestyle="--"),
        mpf.make_addplot(rsi_50, panel=2, width=0.8, linestyle=":"),
        mpf.make_addplot(rsi_30, panel=2, width=0.8, linestyle="--"),
    ]

    title = (
        f"{ticker} | Bollinger Bands({bb_period}, {bb_std}) + RSI({rsi_period})\n"
        f"^ Buy Rebound | v Sell Reversion | * Breakout Up | X Breakout Down"
    )

    fig, _ = mpf.plot(
        df,
        type="candle",
        style="yahoo",
        addplot=addplots,
        volume=volume,
        panel_ratios=(6, 2, 2),
        figscale=1.2,
        figratio=(16, 9),
        title=title,
        returnfig=True,
        tight_layout=True,
    )

    today_str = datetime.now().strftime("%Y%m%d")
    png_path = output_dir / f"{today_str}_{ticker.lower()}_bollinger_rsi.png"
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"정적 차트 저장 완료: {png_path}")
    return png_path


def save_interactive_chart(
    df: pd.DataFrame,
    ticker: str,
    output_dir: Path,
):
    """
    Plotly 동적 HTML 차트 저장
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    df_plot = df.copy()

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        row_heights=[0.60, 0.18, 0.22],
        subplot_titles=(
            f"{ticker} Price + Bollinger Bands + Signals",
            "Volume",
            "RSI"
        )
    )

    # Candlestick
    fig.add_trace(
        go.Candlestick(
            x=df_plot.index,
            open=df_plot["Open"],
            high=df_plot["High"],
            low=df_plot["Low"],
            close=df_plot["Close"],
            name="Price",
            increasing_line_width=1,
            decreasing_line_width=1,
        ),
        row=1, col=1
    )

    # Bollinger Bands
    fig.add_trace(
        go.Scatter(
            x=df_plot.index,
            y=df_plot["UB"],
            mode="lines",
            name="Upper Band",
            line=dict(width=1),
            hovertemplate="Date=%{x}<br>UB=%{y:.2f}<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df_plot.index,
            y=df_plot["MB"],
            mode="lines",
            name="Middle Band",
            line=dict(width=1),
            hovertemplate="Date=%{x}<br>MB=%{y:.2f}<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df_plot.index,
            y=df_plot["LB"],
            mode="lines",
            name="Lower Band",
            line=dict(width=1),
            hovertemplate="Date=%{x}<br>LB=%{y:.2f}<extra></extra>"
        ),
        row=1, col=1
    )

    # Signals
    buy_df = df_plot[df_plot["BUY_REBOUND"]]
    sell_df = df_plot[df_plot["SELL_REVERSION"]]
    breakout_up_df = df_plot[df_plot["BREAKOUT_UP"]]
    breakout_down_df = df_plot[df_plot["BREAKOUT_DOWN"]]

    fig.add_trace(
        go.Scatter(
            x=buy_df.index,
            y=buy_df["Low"] * 0.985,
            mode="markers",
            name="BUY_REBOUND",
            marker=dict(symbol="triangle-up", size=10),
            hovertemplate="Date=%{x}<br>BUY_REBOUND<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=sell_df.index,
            y=sell_df["High"] * 1.015,
            mode="markers",
            name="SELL_REVERSION",
            marker=dict(symbol="triangle-down", size=10),
            hovertemplate="Date=%{x}<br>SELL_REVERSION<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=breakout_up_df.index,
            y=breakout_up_df["Low"] * 0.975,
            mode="markers",
            name="BREAKOUT_UP",
            marker=dict(symbol="star", size=11),
            hovertemplate="Date=%{x}<br>BREAKOUT_UP<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=breakout_down_df.index,
            y=breakout_down_df["High"] * 1.025,
            mode="markers",
            name="BREAKOUT_DOWN",
            marker=dict(symbol="x", size=10),
            hovertemplate="Date=%{x}<br>BREAKOUT_DOWN<extra></extra>"
        ),
        row=1, col=1
    )

    # Volume
    fig.add_trace(
        go.Bar(
            x=df_plot.index,
            y=df_plot["Volume"],
            name="Volume",
            hovertemplate="Date=%{x}<br>Volume=%{y}<extra></extra>"
        ),
        row=2, col=1
    )

    # RSI
    fig.add_trace(
        go.Scatter(
            x=df_plot.index,
            y=df_plot["RSI"],
            mode="lines",
            name="RSI",
            line=dict(width=1),
            hovertemplate="Date=%{x}<br>RSI=%{y:.2f}<extra></extra>"
        ),
        row=3, col=1
    )

    fig.add_hline(y=70, row=3, col=1, line_dash="dash")
    fig.add_hline(y=50, row=3, col=1, line_dash="dot")
    fig.add_hline(y=30, row=3, col=1, line_dash="dash")

    fig.update_layout(
        title=f"{ticker} Interactive Bollinger Bands + RSI Chart",
        xaxis_rangeslider_visible=False,
        height=950,
        hovermode="x unified",
        legend=dict(orientation="h"),
    )

    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    fig.update_yaxes(title_text="RSI", row=3, col=1, range=[0, 100])

    today_str = datetime.now().strftime("%Y%m%d")
    html_path = output_dir / f"{today_str}_{ticker.lower()}_interactive.html"
    fig.write_html(html_path, auto_open=False)

    print(f"동적 차트 저장 완료: {html_path}")
    return html_path


def summarize_signals(df: pd.DataFrame):
    print("\n[신호 개수]")
    print(f"BUY_REBOUND    : {int(df['BUY_REBOUND'].sum())}")
    print(f"SELL_REVERSION : {int(df['SELL_REVERSION'].sum())}")
    print(f"BREAKOUT_UP    : {int(df['BREAKOUT_UP'].sum())}")
    print(f"BREAKOUT_DOWN  : {int(df['BREAKOUT_DOWN'].sum())}")

    signal_cols = ["BUY_REBOUND", "SELL_REVERSION", "BREAKOUT_UP", "BREAKOUT_DOWN"]
    recent_signals = df.loc[
        df[signal_cols].any(axis=1),
        ["Open", "High", "Low", "Close", "Volume", "RSI"] + signal_cols
    ]

    if recent_signals.empty:
        print("\n신호가 없습니다.")
    else:
        print("\n[최근 신호]")
        print(recent_signals.tail(15))


def run_analysis(
    ticker: str = "RXRX",
    period: str = "3y",
    interval: str = "1d",
    bb_period: int = 20,
    bb_std: float = 2.0,
    rsi_period: int = 14,
    volume: bool = True,
    output_dir: str = "output",
    open_html: bool = True,
):
    """
    전체 실행 함수
    """
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"데이터 다운로드 중: {ticker}")
    raw_df = download_data(ticker=ticker, period=period, interval=interval)

    print("지표 계산 중...")
    df = add_indicators(
        raw_df,
        bb_period=bb_period,
        bb_std=bb_std,
        rsi_period=rsi_period,
    )

    csv_path = save_csv_with_today(df, ticker, out_dir)
    png_path = save_static_chart(df, ticker, out_dir, bb_period, bb_std, rsi_period, volume=volume)
    html_path = save_interactive_chart(df, ticker, out_dir)

    summarize_signals(df)

    if open_html:
        webbrowser.open(html_path.resolve().as_uri())

    return {
        "data": df,
        "csv_path": csv_path,
        "png_path": png_path,
        "html_path": html_path,
    }

In [7]:
run_analysis(
    ticker="RXRX",
    period="3y",
    interval="1d",
    bb_period=20,
    bb_std=2.0,
    rsi_period=14,
    volume=True,
    output_dir="output",
    open_html=True
)

데이터 다운로드 중: RXRX
지표 계산 중...
CSV 저장 완료: output/20260310_rxrx.csv


/Users/lol/Documents/GitHub/.venv/lib/python3.14/site-packages/mplfinance/_arg_validators.py:84: UserWarning: 


            POSSIBLE TO SEE DETAILS (Candles, Ohlc-Bars, Etc.)
   For more information see:
   - https://github.com/matplotlib/mplfinance/wiki/Plotting-Too-Much-Data
   
   TO SILENCE THIS WARNING, set `type='line'` in `mpf.plot()`
   OR set kwarg `warn_too_much_data=N` where N is an integer 
   LARGER than the number of data points you want to plot.

  warnings.warn('\n\n ================================================================= '+


정적 차트 저장 완료: output/20260310_rxrx_bollinger_rsi.png
동적 차트 저장 완료: output/20260310_rxrx_interactive.html

[신호 개수]
BUY_REBOUND    : 1
SELL_REVERSION : 7
BREAKOUT_UP    : 16
BREAKOUT_DOWN  : 11

[최근 신호]
             Open   High    Low  Close    Volume        RSI  BUY_REBOUND  \
Date                                                                       
2024-11-11  7.500  8.490  7.230  7.840  15577600  68.381374        False   
2025-03-31  5.495  5.510  5.190  5.290  17981300  33.283070        False   
2025-06-04  4.450  5.080  4.410  4.910  46074900  54.916095        False   
2025-06-06  4.720  5.520  4.705  5.490  60323000  61.380519        False   
2025-07-09  5.460  5.770  5.430  5.590  34721700  59.223622        False   
2025-07-10  5.710  5.879  5.570  5.715  28683900  60.995224        False   
2025-07-18  5.650  6.225  5.570  5.840  53043400  61.592297        False   
2025-07-21  6.110  7.150  6.030  6.400  75466000  68.438967        False   
2025-08-20  5.010  5.090  4.730  4.770  2

{'data':             Open  High    Low  Close    Volume      MB       STD        UB  \
 Date                                                                         
 2023-03-10  7.44  7.48  6.755   7.07   1862600     NaN       NaN       NaN   
 2023-03-13  6.98  7.75  6.870   7.67   1486600     NaN       NaN       NaN   
 2023-03-14  7.95  7.95  7.370   7.63   1046900     NaN       NaN       NaN   
 2023-03-15  7.43  7.60  7.130   7.48   1146800     NaN       NaN       NaN   
 2023-03-16  7.71  7.75  7.330   7.65    916600     NaN       NaN       NaN   
 ...          ...   ...    ...    ...       ...     ...       ...       ...   
 2026-03-03  3.46  3.61  3.440   3.54  12569200  3.6800  0.196087  4.072173   
 2026-03-04  3.60  3.76  3.530   3.64  16256000  3.6570  0.170824  3.998649   
 2026-03-05  3.55  3.68  3.460   3.54  14054500  3.6390  0.163061  3.965123   
 2026-03-06  3.47  3.53  3.380   3.46  12183800  3.6340  0.166895  3.967790   
 2026-03-09  3.33  3.54  3.290   3.51  12728